
# Roxy notebook example: Motif and pattern descriptors

This notebook is a **reference implementation example** for the **motif and pattern descriptor family** in Roxy.

Pattern-based descriptors are powerful because they capture **specific biological sequence signatures** that are not always well summarized by composition alone.

## Covered outputs

This notebook implements:

- sequence cleaning
- predefined motif patterns
- user-defined regex patterns
- presence / absence descriptors
- motif counts
- motif density normalized by sequence length
- optional N-terminal and C-terminal motif occurrence
- simple class-style implementation for later migration into Roxy

The goal is to provide a **clean teaching implementation** that can later become the real pattern-based descriptor module in Roxy.


In [1]:

import re
import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "motif_1",
            "motif_2",
            "motif_3",
            "motif_4",
            "motif_5",
            "motif_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,motif_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,motif_2,GGGGGGGGGGGGGGG,B
2,motif_3,KRRKRRKRRKRRDDDDEE,A
3,motif_4,ACDEFGHIKLMNPQRSTVWY,B
4,motif_5,PPPPGSSSSSTTTTNNQQQ,A
5,motif_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and pattern library

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Predefined motif library for demonstration purposes.
# These are intentionally simple and interpretable examples.
PREDEFINED_PATTERNS = {
    "CxxC": r"C.{2}C",
    "PxxP": r"P.{2}P",
    "polyK_3plus": r"K{3,}",
    "polyR_3plus": r"R{3,}",
    "acidic_patch_3plus": r"[DE]{3,}",
    "basic_patch_3plus": r"[KRH]{3,}",
    "gly_rich_4plus": r"(?:G.*){4,}",
    "proline_rich_4plus": r"(?:P.*){4,}",
    "ser_thr_rich_4plus": r"(?:[ST].*){4,}",
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    """Keep only the 20 standard amino acids."""
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def count_pattern_matches(seq: str, pattern: str) -> int:
    """Count non-overlapping regex matches."""
    if len(seq) == 0:
        return 0
    return len(re.findall(pattern, seq))


def has_pattern(seq: str, pattern: str) -> int:
    """Binary presence/absence for a regex pattern."""
    return int(count_pattern_matches(seq, pattern) > 0)


def pattern_density(seq: str, pattern: str) -> float:
    """Normalize motif count by sequence length."""
    if len(seq) == 0:
        return np.nan
    return count_pattern_matches(seq, pattern) / len(seq)


def terminal_has_pattern(seq: str, pattern: str, side: str = "N", window: int = 10) -> int:
    """Check motif presence in an N- or C-terminal window."""
    if len(seq) == 0:
        return 0
    if side == "N":
        subseq = seq[:window]
    elif side == "C":
        subseq = seq[-window:]
    else:
        raise ValueError("side must be 'N' or 'C'")
    return has_pattern(subseq, pattern)


## Core motif descriptor function

In [5]:

def motif_pattern_descriptors(
    seq: str,
    pattern_dict: dict,
    include_terminal: bool = True,
    terminal_window: int = 10,
) -> dict:
    seq = clean_sequence(seq)

    out = {
        "motif_length": len(seq),
        "motif_valid_residue_count": len(seq),
    }

    for motif_name, pattern in pattern_dict.items():
        out[f"motif_{motif_name}_present"] = has_pattern(seq, pattern)
        out[f"motif_{motif_name}_count"] = count_pattern_matches(seq, pattern)
        out[f"motif_{motif_name}_density"] = pattern_density(seq, pattern)

        if include_terminal:
            out[f"motif_{motif_name}_nterm_present_w{terminal_window}"] = terminal_has_pattern(
                seq, pattern, side="N", window=terminal_window
            )
            out[f"motif_{motif_name}_cterm_present_w{terminal_window}"] = terminal_has_pattern(
                seq, pattern, side="C", window=terminal_window
            )

    return out


## Functional usage with predefined patterns

In [6]:

example = motif_pattern_descriptors(
    df_demo.loc[0, "sequence"],
    PREDEFINED_PATTERNS,
    include_terminal=True,
    terminal_window=10,
)
list(example.items())[:15]


[('motif_length', 24),
 ('motif_valid_residue_count', 24),
 ('motif_CxxC_present', 0),
 ('motif_CxxC_count', 0),
 ('motif_CxxC_density', 0.0),
 ('motif_CxxC_nterm_present_w10', 0),
 ('motif_CxxC_cterm_present_w10', 0),
 ('motif_PxxP_present', 0),
 ('motif_PxxP_count', 0),
 ('motif_PxxP_density', 0.0),
 ('motif_PxxP_nterm_present_w10', 0),
 ('motif_PxxP_cterm_present_w10', 0),
 ('motif_polyK_3plus_present', 0),
 ('motif_polyK_3plus_count', 0),
 ('motif_polyK_3plus_density', 0.0)]

## Apply predefined motif descriptors to the full dataset

In [7]:

df_motif = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: motif_pattern_descriptors(
                x,
                PREDEFINED_PATTERNS,
                include_terminal=True,
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_motif.head()


,sequence_id,sequence,label,motif_length,motif_valid_residue_count,motif_CxxC_present,motif_CxxC_count,motif_CxxC_density,motif_CxxC_nterm_present_w10,motif_CxxC_cterm_present_w10,...,motif_proline_rich_4plus_present,motif_proline_rich_4plus_count,motif_proline_rich_4plus_density,motif_proline_rich_4plus_nterm_present_w10,motif_proline_rich_4plus_cterm_present_w10,motif_ser_thr_rich_4plus_present,motif_ser_thr_rich_4plus_count,motif_ser_thr_rich_4plus_density,motif_ser_thr_rich_4plus_nterm_present_w10,motif_ser_thr_rich_4plus_cterm_present_w10
0,motif_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,0.041667,0.0,0.0
1,motif_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,motif_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,motif_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,motif_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.052632,1.0,0.0,1.0,1.0,0.052632,1.0,1.0


## Inspect motif descriptor columns

In [8]:

motif_cols = [c for c in df_motif.columns if c.startswith("motif_") and c not in {"motif_length", "motif_valid_residue_count"}]
len(motif_cols), motif_cols[:15]


(45,
 ['motif_CxxC_present',
  'motif_CxxC_count',
  'motif_CxxC_density',
  'motif_CxxC_nterm_present_w10',
  'motif_CxxC_cterm_present_w10',
  'motif_PxxP_present',
  'motif_PxxP_count',
  'motif_PxxP_density',
  'motif_PxxP_nterm_present_w10',
  'motif_PxxP_cterm_present_w10',
  'motif_polyK_3plus_present',
  'motif_polyK_3plus_count',
  'motif_polyK_3plus_density',
  'motif_polyK_3plus_nterm_present_w10',
  'motif_polyK_3plus_cterm_present_w10'])

In [9]:

df_motif[
    [
        "sequence_id",
        "motif_CxxC_present",
        "motif_PxxP_present",
        "motif_polyK_3plus_present",
        "motif_polyR_3plus_present",
        "motif_acidic_patch_3plus_present",
        "motif_basic_patch_3plus_present",
    ]
]


,sequence_id,motif_CxxC_present,motif_PxxP_present,motif_polyK_3plus_present,motif_polyR_3plus_present,motif_acidic_patch_3plus_present,motif_basic_patch_3plus_present
0,motif_1,0.0,0.0,0.0,0.0,0.0,0.0
1,motif_2,0.0,0.0,0.0,0.0,0.0,0.0
2,motif_3,0.0,0.0,0.0,0.0,1.0,1.0
3,motif_4,0.0,0.0,0.0,0.0,0.0,0.0
4,motif_5,0.0,1.0,0.0,0.0,0.0,0.0
5,motif_6,0.0,0.0,0.0,0.0,0.0,0.0


## User-defined patterns

In [10]:

USER_PATTERNS = {
    "double_gly": r"GG",
    "double_pro": r"PP",
    "kr_pair": r"KR",
    "acidic_pair": r"DE|ED|DD|EE",
}

USER_PATTERNS


{'double_gly': 'GG',
 'double_pro': 'PP',
 'kr_pair': 'KR',
 'acidic_pair': 'DE|ED|DD|EE'}

In [11]:

df_user_motif = pd.concat(
    [
        df_demo[["sequence_id", "sequence"]],
        df_demo["sequence"].apply(
            lambda x: motif_pattern_descriptors(
                x,
                USER_PATTERNS,
                include_terminal=False,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_user_motif.head()


,sequence_id,sequence,motif_length,motif_valid_residue_count,motif_double_gly_present,motif_double_gly_count,motif_double_gly_density,motif_double_pro_present,motif_double_pro_count,motif_double_pro_density,motif_kr_pair_present,motif_kr_pair_count,motif_kr_pair_density,motif_acidic_pair_present,motif_acidic_pair_count,motif_acidic_pair_density
0,motif_1,MKWVTFISLLFLFSSAYSRGVFRR,24.0,24.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000
1,motif_2,GGGGGGGGGGGGGGG,15.0,15.0,1.0,7.0,0.466667,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000
2,motif_3,KRRKRRKRRKRRDDDDEE,18.0,18.0,0.0,0.0,0.000000,0.0,0.0,0.000000,1.0,4.0,0.222222,1.0,3.0,0.166667
3,motif_4,ACDEFGHIKLMNPQRSTVWY,20.0,20.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,1.0,1.0,0.050000
4,motif_5,PPPPGSSSSSTTTTNNQQQ,19.0,19.0,0.0,0.0,0.000000,1.0,2.0,0.105263,0.0,0.0,0.000000,0.0,0.0,0.000000


## Dataset-level motif summary

In [12]:

motif_summary = (
    df_motif[motif_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

motif_summary.head(15)


,descriptor,mean_value
0,motif_ser_thr_rich_4plus_present,0.333333
1,motif_ser_thr_rich_4plus_count,0.333333
2,motif_PxxP_nterm_present_w10,0.166667
3,motif_PxxP_count,0.166667
4,motif_PxxP_present,0.166667
5,motif_basic_patch_3plus_nterm_present_w10,0.166667
6,motif_acidic_patch_3plus_cterm_present_w10,0.166667
7,motif_acidic_patch_3plus_count,0.166667
8,motif_basic_patch_3plus_present,0.166667
9,motif_basic_patch_3plus_count,0.166667


## Sanity checks

In [13]:

assert "motif_CxxC_present" in df_motif.columns
assert "motif_PxxP_count" in df_motif.columns
assert "motif_polyK_3plus_density" in df_motif.columns
assert "motif_basic_patch_3plus_nterm_present_w10" in df_motif.columns
assert df_motif["motif_length"].min() > 0

print(f"Number of motif/pattern descriptor columns: {len(motif_cols)}")
print("Motif and pattern descriptor checks passed.")


Number of motif/pattern descriptor columns: 45
Motif and pattern descriptor checks passed.


## Class-style implementation closer to the real package

In [14]:

class PatternDescriptors:
    """Example class-style motif/pattern implementation for later migration into Roxy."""

    def __init__(self, pattern_dict=None, include_terminal=True, terminal_window=10):
        self.pattern_dict = PREDEFINED_PATTERNS if pattern_dict is None else dict(pattern_dict)
        self.include_terminal = include_terminal
        self.terminal_window = terminal_window

    def transform_sequence(self, seq: str) -> dict:
        return motif_pattern_descriptors(
            seq,
            self.pattern_dict,
            include_terminal=self.include_terminal,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


pattern_transformer = PatternDescriptors(
    pattern_dict=PREDEFINED_PATTERNS,
    include_terminal=True,
    terminal_window=10,
)

pattern_matrix = pattern_transformer.transform(df_demo["sequence"].tolist())
pattern_matrix.head()


,motif_length,motif_valid_residue_count,motif_CxxC_present,motif_CxxC_count,motif_CxxC_density,motif_CxxC_nterm_present_w10,motif_CxxC_cterm_present_w10,motif_PxxP_present,motif_PxxP_count,motif_PxxP_density,...,motif_proline_rich_4plus_present,motif_proline_rich_4plus_count,motif_proline_rich_4plus_density,motif_proline_rich_4plus_nterm_present_w10,motif_proline_rich_4plus_cterm_present_w10,motif_ser_thr_rich_4plus_present,motif_ser_thr_rich_4plus_count,motif_ser_thr_rich_4plus_density,motif_ser_thr_rich_4plus_nterm_present_w10,motif_ser_thr_rich_4plus_cterm_present_w10
0,24,24,0,0,0.0,0,0,0,0,0.000000,...,0,0,0.000000,0,0,1,1,0.041667,0,0
1,15,15,0,0,0.0,0,0,0,0,0.000000,...,0,0,0.000000,0,0,0,0,0.000000,0,0
2,18,18,0,0,0.0,0,0,0,0,0.000000,...,0,0,0.000000,0,0,0,0,0.000000,0,0
3,20,20,0,0,0.0,0,0,0,0,0.000000,...,0,0,0.000000,0,0,0,0,0.000000,0,0
4,19,19,0,0,0.0,0,0,1,1,0.052632,...,1,1,0.052632,1,0,1,1,0.052632,1,1


## Merge transformer output back to the dataset

In [15]:

df_motif_class = pd.concat([df_demo, pattern_matrix], axis=1)
df_motif_class.head()


,sequence_id,sequence,label,motif_length,motif_valid_residue_count,motif_CxxC_present,motif_CxxC_count,motif_CxxC_density,motif_CxxC_nterm_present_w10,motif_CxxC_cterm_present_w10,...,motif_proline_rich_4plus_present,motif_proline_rich_4plus_count,motif_proline_rich_4plus_density,motif_proline_rich_4plus_nterm_present_w10,motif_proline_rich_4plus_cterm_present_w10,motif_ser_thr_rich_4plus_present,motif_ser_thr_rich_4plus_count,motif_ser_thr_rich_4plus_density,motif_ser_thr_rich_4plus_nterm_present_w10,motif_ser_thr_rich_4plus_cterm_present_w10
0,motif_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0,0,0.0,0,0,...,0,0,0.000000,0,0,1,1,0.041667,0,0
1,motif_2,GGGGGGGGGGGGGGG,B,15,15,0,0,0.0,0,0,...,0,0,0.000000,0,0,0,0,0.000000,0,0
2,motif_3,KRRKRRKRRKRRDDDDEE,A,18,18,0,0,0.0,0,0,...,0,0,0.000000,0,0,0,0,0.000000,0,0
3,motif_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0,0,0.0,0,0,...,0,0,0.000000,0,0,0,0,0.000000,0,0
4,motif_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0,0,0.0,0,0,...,1,1,0.052632,1,0,1,1,0.052632,1,1



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move predefined motif libraries into `roxy/core/constants.py` or `roxy/sequence/patterns.py`
- keep the regex matching logic inside `roxy/sequence/patterns.py`
- expose a class such as `PatternDescriptors`
- allow configurable:
  - predefined motif libraries
  - user-defined motif dictionaries
  - terminal windows
  - presence-only vs count+density modes
- add tests for:
  - empty sequences
  - lower-case input
  - overlapping motif edge cases
  - sequences with no motif hits
  - strongly repetitive sequences


## Optional export

In [ ]:
# df_motif.to_csv("demo_motif_pattern_descriptors.csv", index=False)
